In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os

BASE_DIR = "/content/drive/MyDrive/JoshTalks_ASR"

TRAIN_CSV = os.path.join(BASE_DIR, "03_processed", "train_val_split", "train.csv")
VAL_CSV = os.path.join(BASE_DIR, "03_processed", "train_val_split", "val.csv")

MODEL_DIR = os.path.join(BASE_DIR, "04_models", "whisper_small_finetuned")
os.makedirs(MODEL_DIR, exist_ok=True)

print("Train CSV:", TRAIN_CSV)
print("Val CSV:", VAL_CSV)
print("Model dir:", MODEL_DIR)
print("Train exists:", os.path.exists(TRAIN_CSV))
print("Val exists:", os.path.exists(VAL_CSV))

Train CSV: /content/drive/MyDrive/JoshTalks_ASR/03_processed/train_val_split/train.csv
Val CSV: /content/drive/MyDrive/JoshTalks_ASR/03_processed/train_val_split/val.csv
Model dir: /content/drive/MyDrive/JoshTalks_ASR/04_models/whisper_small_finetuned
Train exists: True
Val exists: True


In [6]:
!pip install -q transformers datasets evaluate jiwer librosa soundfile accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 67.4 MB/s eta 0:00:00


In [7]:
import os
import pandas as pd
import numpy as np
import torch
import librosa

from datasets import Dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

In [8]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [9]:
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)

train_df.head()

Train shape: (93, 5)
Val shape: (11, 5)


,recording_id,user_id,duration,audio_path,text
0,615319,266660,452,/content/drive/MyDrive/JoshTalks_ASR/02_downlo...,हाँ हां तो ठीक है मैं मै अपनी जो घटना है मैं ब...
1,857737,212988,1079,/content/drive/MyDrive/JoshTalks_ASR/02_downlo...,मुझे भी एक बेबी पसन्द था जी मेरा आवाज आ रही है...
2,270037,218790,836,/content/drive/MyDrive/JoshTalks_ASR/02_downlo...,हूं हूं नहीं नहीं यार किसी को तो नहीं होते आज ...
3,489638,25489,642,/content/drive/MyDrive/JoshTalks_ASR/02_downlo...,है ना हा.और ऐसी चीजे है ना अपना मतलब पूरा जीवन...
4,526266,286851,522,/content/drive/MyDrive/JoshTalks_ASR/02_downlo...,हां जी पहले बात करते हैं विवाह की तो इस मुवी म...


In [10]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

print(train_dataset)
print(val_dataset)

Dataset({
    features: ['recording_id', 'user_id', 'duration', 'audio_path', 'text'],
    num_rows: 93
})
Dataset({
    features: ['recording_id', 'user_id', 'duration', 'audio_path', 'text'],
    num_rows: 11
})


In [14]:
train_dataset = train_dataset.cast_column("audio_path", Audio(sampling_rate=16000))
val_dataset = val_dataset.cast_column("audio_path", Audio(sampling_rate=16000))

print(train_dataset[0].keys())
print(train_dataset[0]["audio_path"].keys())
print("Sampling rate:", train_dataset[0]["audio_path"]["sampling_rate"])
print("Text:", train_dataset[0]["text"][:200])

IndexError: Invalid key: 0 is out of bounds for size 0

In [15]:
print("train_df shape:", train_df.shape)
print("val_df shape:", val_df.shape)

print("\nTrain columns:", train_df.columns.tolist())
print("\nFirst 3 audio paths:")
print(train_df["audio_path"].head(3).tolist())

print("\nDo first 3 files exist?")
for p in train_df["audio_path"].head(3):
    print(p, "->", os.path.exists(p))

train_df shape: (0, 6)
val_df shape: (0, 6)

Train columns: ['recording_id', 'user_id', 'duration', 'audio_path', 'text', 'char_len']

First 3 audio paths:
[]

Do first 3 files exist?


In [16]:
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print(train_df.columns.tolist())
train_df.head()

Train shape: (93, 5)
Val shape: (11, 5)
['recording_id', 'user_id', 'duration', 'audio_path', 'text']


,recording_id,user_id,duration,audio_path,text
0,615319,266660,452,/content/drive/MyDrive/JoshTalks_ASR/02_downlo...,हाँ हां तो ठीक है मैं मै अपनी जो घटना है मैं ब...
1,857737,212988,1079,/content/drive/MyDrive/JoshTalks_ASR/02_downlo...,मुझे भी एक बेबी पसन्द था जी मेरा आवाज आ रही है...
2,270037,218790,836,/content/drive/MyDrive/JoshTalks_ASR/02_downlo...,हूं हूं नहीं नहीं यार किसी को तो नहीं होते आज ...
3,489638,25489,642,/content/drive/MyDrive/JoshTalks_ASR/02_downlo...,है ना हा.और ऐसी चीजे है ना अपना मतलब पूरा जीवन...
4,526266,286851,522,/content/drive/MyDrive/JoshTalks_ASR/02_downlo...,हां जी पहले बात करते हैं विवाह की तो इस मुवी म...


In [18]:
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))

print(train_dataset)
print(val_dataset)

Dataset({
    features: ['recording_id', 'user_id', 'duration', 'audio_path', 'text'],
    num_rows: 93
})
Dataset({
    features: ['recording_id', 'user_id', 'duration', 'audio_path', 'text'],
    num_rows: 11
})


In [20]:
train_dataset = train_dataset.cast_column("audio_path", Audio(sampling_rate=16000))
val_dataset = val_dataset.cast_column("audio_path", Audio(sampling_rate=16000))

sample = train_dataset[0]
print(sample.keys())
# print(sample["audio_path"].keys()) # This line caused the error
print("Sampling rate:", sample["audio_path"]["sampling_rate"])
print("Text preview:", sample["text"][:150])

dict_keys(['recording_id', 'user_id', 'duration', 'audio_path', 'text'])
Sampling rate: 16000
Text preview: हाँ हां तो ठीक है मैं मै अपनी जो घटना है मैं बताता हूँ देखिए जेसे हां तो इसके बाद है हमारे पास एक है म मैंने जो अ अ किताबों में पढ़ा था और हमारे दस्वी


In [21]:
processor = WhisperProcessor.from_pretrained(
    "openai/whisper-small",
    language="Hindi",
    task="transcribe"
)

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
model.to(device)

model.generation_config.language = "hindi"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

print("Model loaded on:", device)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Model loaded on: cuda


In [22]:
def prepare_example(batch):
    audio = batch["audio_path"]

    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

In [23]:
train_prepared = train_dataset.map(
    prepare_example,
    remove_columns=train_dataset.column_names
)

val_prepared = val_dataset.map(
    prepare_example,
    remove_columns=val_dataset.column_names
)

print(train_prepared)
print(val_prepared)

Map:   0%|          | 0/93 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (2290 > 1024). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/11 [00:00<?, ? examples/s]

Dataset({
    features: ['input_features', 'labels'],
    num_rows: 93
})
Dataset({
    features: ['input_features', 'labels'],
    num_rows: 11
})


In [24]:
def add_label_len(example):
    example["label_len"] = len(example["labels"])
    return example

train_prepared = train_prepared.map(add_label_len)
val_prepared = val_prepared.map(add_label_len)

print("Train max label len:", max(train_prepared["label_len"]))
print("Val max label len:", max(val_prepared["label_len"]))

print("\nTrain label length stats:")
print(pd.Series(train_prepared["label_len"]).describe())

Map:   0%|          | 0/93 [00:00<?, ? examples/s]

Map:   0%|          | 0/11 [00:00<?, ? examples/s]

Train max label len: 12649
Val max label len: 5033

Train label length stats:
count       93.000000
mean      4727.655914
std       2336.305279
min       1180.000000
25%       3067.000000
50%       4102.000000
75%       6585.000000
max      12649.000000
dtype: float64


In [25]:
train_df["char_len"] = train_df["text"].astype(str).apply(len)
val_df["char_len"] = val_df["text"].astype(str).apply(len)

print("Train char length stats:")
print(train_df["char_len"].describe())

print("\nVal char length stats:")
print(val_df["char_len"].describe())

Train char length stats:
count       93.000000
mean      4694.473118
std       2306.555509
min       1202.000000
25%       3078.000000
50%       4055.000000
75%       6461.000000
max      12428.000000
Name: char_len, dtype: float64

Val char length stats:
count      11.000000
mean     3074.090909
std      1059.282630
min      1295.000000
25%      2663.500000
50%      2984.000000
75%      3238.000000
max      4901.000000
Name: char_len, dtype: float64


In [26]:
short_train_df = train_df[train_df["char_len"] <= 2000].copy()
short_val_df = val_df[val_df["char_len"] <= 2000].copy()

print("Short train shape:", short_train_df.shape)
print("Short val shape:", short_val_df.shape)

print("\nShort train char stats:")
print(short_train_df["char_len"].describe())

print("\nShort val char stats:")
print(short_val_df["char_len"].describe())

Short train shape: (5, 6)
Short val shape: (1, 6)

Short train char stats:
count       5.000000
mean     1563.800000
std       302.070687
min      1202.000000
25%      1307.000000
50%      1677.000000
75%      1699.000000
max      1934.000000
Name: char_len, dtype: float64

Short val char stats:
count       1.0
mean     1295.0
std         NaN
min      1295.0
25%      1295.0
50%      1295.0
75%      1295.0
max      1295.0
Name: char_len, dtype: float64


In [27]:
import json
import os

transcript_dir = os.path.join(BASE_DIR, "02_downloaded", "transcripts")

sample_json_path = os.path.join(transcript_dir, "825780_transcription.json")

with open(sample_json_path, "r", encoding="utf-8") as f:
    sample_segments = json.load(f)

print("Type:", type(sample_segments))
print("Number of segments:", len(sample_segments))
print("\nFirst 3 segments:\n")
for seg in sample_segments[:3]:
    print(seg)

Type: <class 'list'>
Number of segments: 24

First 3 segments:

{'start': 0.11, 'end': 14.42, 'speaker_id': 245746, 'text': 'अब काफी अच्छा होता है क्योंकि उनकी जनसंख्या बहुत कम दी जा रही है तो हमें उनको देखना था तो एक देखना था मतलब वो तो देखना था लेकिन हमारा प्रोजेक्ट भी था कि जो जन जाती पाई जाती है उधर कि उधर की एरिया में उसके बारे में देखना अब'}
{'start': 14.42, 'end': 29.03, 'speaker_id': 245746, 'text': 'अनुभव करके कुछ लिखना था तो वह तो बिना देखिए नहीं हो सकती थी तो हम वहां गया थे कुड़रमा घाटी तरफ पर दिवोग काफी जंगली एरिया है वह जो खांड जनजाति पाए जाती ना वहां पाए जाती है तो'}
{'start': 29.03, 'end': 41.84, 'speaker_id': 245746, 'text': 'जंगल का सफर होता है जब हम रहने के लिए गए थे नातो चाहते के साथ जैसे हम वहाँ पहले एंटर किये थे तो पहले तो गिर गया थे लगड़ा के उपर से नीचे'}


In [28]:
import soundfile as sf

SEGMENT_AUDIO_DIR = os.path.join(BASE_DIR, "03_processed", "segmented_audio")
os.makedirs(SEGMENT_AUDIO_DIR, exist_ok=True)

print("Segment audio dir:", SEGMENT_AUDIO_DIR)

Segment audio dir: /content/drive/MyDrive/JoshTalks_ASR/03_processed/segmented_audio


In [29]:
def segment_one_recording(recording_id, min_text_len=5, max_segment_sec=30):
    audio_path = os.path.join(BASE_DIR, "02_downloaded", "audio", f"{recording_id}.wav")
    transcript_path = os.path.join(BASE_DIR, "02_downloaded", "transcripts", f"{recording_id}_transcription.json")

    if not os.path.exists(audio_path):
        return []
    if not os.path.exists(transcript_path):
        return []

    with open(transcript_path, "r", encoding="utf-8") as f:
        segments = json.load(f)

    audio, sr = librosa.load(audio_path, sr=16000)

    rows = []

    for i, seg in enumerate(segments):
        start = float(seg.get("start", 0))
        end = float(seg.get("end", 0))
        text = str(seg.get("text", "")).strip()

        if len(text) < min_text_len:
            continue
        if end <= start:
            continue
        if (end - start) > max_segment_sec:
            continue

        start_idx = int(start * sr)
        end_idx = int(end * sr)
        chunk = audio[start_idx:end_idx]

        if len(chunk) == 0:
            continue

        seg_filename = f"{recording_id}_seg_{i:03d}.wav"
        seg_path = os.path.join(SEGMENT_AUDIO_DIR, seg_filename)

        sf.write(seg_path, chunk, sr)

        rows.append({
            "recording_id": recording_id,
            "segment_id": i,
            "audio_path": seg_path,
            "text": text,
            "start": start,
            "end": end,
            "duration_sec": end - start,
            "char_len": len(text)
        })

    return rows

In [30]:
one_seg_rows = segment_one_recording(825780)

print("Number of segments created:", len(one_seg_rows))

seg_test_df = pd.DataFrame(one_seg_rows)
seg_test_df.head()

Number of segments created: 21


,recording_id,segment_id,audio_path,text,start,end,duration_sec,char_len
0,825780,0,/content/drive/MyDrive/JoshTalks_ASR/03_proces...,अब काफी अच्छा होता है क्योंकि उनकी जनसंख्या बह...,0.11,14.42,14.31,222
1,825780,1,/content/drive/MyDrive/JoshTalks_ASR/03_proces...,अनुभव करके कुछ लिखना था तो वह तो बिना देखिए नह...,14.42,29.03,14.61,173
2,825780,2,/content/drive/MyDrive/JoshTalks_ASR/03_proces...,जंगल का सफर होता है जब हम रहने के लिए गए थे ना...,29.03,41.84,12.81,135
3,825780,3,/content/drive/MyDrive/JoshTalks_ASR/03_proces...,पहली बारी था क्योंकि चलना नहीं आता न वहाँ का ज...,42.47,50.57,8.10,106
4,825780,4,/content/drive/MyDrive/JoshTalks_ASR/03_proces...,हां तो फिर वहां जो दिन भर तो दिन में तो खोजने ...,52.70,66.83,14.13,157


In [35]:
all_segment_rows = []

for rid in tqdm(final_df["recording_id"].tolist()):
    rows = segment_one_recording(rid)
    all_segment_rows.extend(rows)

segments_df = pd.DataFrame(all_segment_rows)

print("Total segmented rows:", len(segments_df))
print("Unique recordings covered:", segments_df["recording_id"].nunique())
segments_df.head()

  0%|          | 0/104 [00:00<?, ?it/s]

Total segmented rows: 5063
Unique recordings covered: 104


,recording_id,segment_id,audio_path,text,start,end,duration_sec,char_len
0,615319,0,/content/drive/MyDrive/JoshTalks_ASR/03_proces...,हाँ हां तो ठीक है मैं मै अपनी जो घटना है मैं ब...,7.46,14.99,7.53,65
1,615319,1,/content/drive/MyDrive/JoshTalks_ASR/03_proces...,हां तो इसके बाद है हमारे पास एक है म मैंने जो ...,24.62,39.53,14.91,134
2,615319,2,/content/drive/MyDrive/JoshTalks_ASR/03_proces...,है ना तो उनके जो जीवन शैली में जो घटनाएं हुई थ...,39.92,54.14,14.22,139
3,615319,3,/content/drive/MyDrive/JoshTalks_ASR/03_proces...,"तो आपने कभी इनके बारे में कभी पढ़ा था,",55.73,59.93,4.20,38
4,615319,4,/content/drive/MyDrive/JoshTalks_ASR/03_proces...,हाँ मीरा बाई,63.17,64.13,0.96,12


In [39]:
MANIFEST_DIR = os.path.join(BASE_DIR, "03_processed", "manifests")

segmented_manifest_path = os.path.join(MANIFEST_DIR, "segmented_manifest.csv")
segments_df.to_csv(segmented_manifest_path, index=False)

print("Saved segmented manifest to:", segmented_manifest_path)
print("Total rows:", len(segments_df))

Saved segmented manifest to: /content/drive/MyDrive/JoshTalks_ASR/03_processed/manifests/segmented_manifest.csv
Total rows: 5063


In [40]:
from sklearn.model_selection import train_test_split

seg_train_df, seg_val_df = train_test_split(
    segments_df,
    test_size=0.1,
    random_state=42,
    shuffle=True
)

seg_train_path = os.path.join(BASE_DIR, "03_processed", "train_val_split", "seg_train.csv")
seg_val_path = os.path.join(BASE_DIR, "03_processed", "train_val_split", "seg_val.csv")

seg_train_df.to_csv(seg_train_path, index=False)
seg_val_df.to_csv(seg_val_path, index=False)

print("Segment train shape:", seg_train_df.shape)
print("Segment val shape:", seg_val_df.shape)
print("Saved seg train to:", seg_train_path)
print("Saved seg val to:", seg_val_path)

Segment train shape: (4556, 8)
Segment val shape: (507, 8)
Saved seg train to: /content/drive/MyDrive/JoshTalks_ASR/03_processed/train_val_split/seg_train.csv
Saved seg val to: /content/drive/MyDrive/JoshTalks_ASR/03_processed/train_val_split/seg_val.csv


In [41]:
seg_train_dataset = Dataset.from_pandas(seg_train_df.reset_index(drop=True))
seg_val_dataset = Dataset.from_pandas(seg_val_df.reset_index(drop=True))

seg_train_dataset = seg_train_dataset.cast_column("audio_path", Audio(sampling_rate=16000))
seg_val_dataset = seg_val_dataset.cast_column("audio_path", Audio(sampling_rate=16000))

print(seg_train_dataset)
print(seg_val_dataset)

sample = seg_train_dataset[0]
print("\nSample text:", sample["text"])
print("Sample audio sampling rate:", sample["audio_path"]["sampling_rate"])

Dataset({
    features: ['recording_id', 'segment_id', 'audio_path', 'text', 'start', 'end', 'duration_sec', 'char_len'],
    num_rows: 4556
})
Dataset({
    features: ['recording_id', 'segment_id', 'audio_path', 'text', 'start', 'end', 'duration_sec', 'char_len'],
    num_rows: 507
})

Sample text: बिल्कुल सर बिल्कुल सर हमारी जो दिनचर्या है उसे पूरी सुव्यवस्थित रूप से बनाना चाहिए और उसका पालन भी करना चाहिए ऐसा होता है कि हम टाइम टेबल तो बना लेते हैं यह करना है यह करना है लेकिन कभी कभी उसका पालन नहीं कर
Sample audio sampling rate: 16000


In [42]:
def prepare_example(batch):
    audio = batch["audio_path"]

    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch


In [43]:
seg_train_prepared = seg_train_dataset.map(
    prepare_example,
    remove_columns=seg_train_dataset.column_names
)

seg_val_prepared = seg_val_dataset.map(
    prepare_example,
    remove_columns=seg_val_dataset.column_names
)

print(seg_train_prepared)
print(seg_val_prepared)

Map:   0%|          | 0/4556 [00:00<?, ? examples/s]

Map:   0%|          | 0/507 [00:00<?, ? examples/s]

Dataset({
    features: ['input_features', 'labels'],
    num_rows: 4556
})
Dataset({
    features: ['input_features', 'labels'],
    num_rows: 507
})


In [44]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print("Data collator ready.")

Data collator ready.


In [46]:
training_args = Seq2SeqTrainingArguments(
    output_dir=MODEL_DIR,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=50,
    max_steps=300,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    logging_steps=10,
    predict_with_generate=False,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none"
)

print("Training arguments ready.")

Training arguments ready.


In [48]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=seg_train_prepared,
    eval_dataset=seg_val_prepared,
    data_collator=data_collator,
)

print("Trainer ready.")

Trainer ready.


In [49]:
train_result = trainer.train()
print(train_result)

Step,Training Loss,Validation Loss
50,1.577985,0.758458
100,1.108400,0.594168
150,1.057183,0.519459


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss
50,1.577985,0.758458
100,1.108400,0.594168
150,1.057183,0.519459
200,0.946906,0.473078
250,0.874158,0.441717
300,0.862715,0.430122


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


TrainOutput(global_step=300, training_loss=1.2726758782068888, metrics={'train_runtime': 1819.5238, 'train_samples_per_second': 1.319, 'train_steps_per_second': 0.165, 'total_flos': 6.92604960768e+17, 'train_loss': 1.2726758782068888, 'epoch': 0.5267778753292361})


In [50]:
import os

seg_dir = os.path.join(BASE_DIR, "03_processed", "segmented_audio")
model_dir = os.path.join(BASE_DIR, "04_models", "whisper_small_finetuned")
down_trans_dir = os.path.join(BASE_DIR, "02_downloaded", "transcripts")

print("Segmented audio files:", len(os.listdir(seg_dir)) if os.path.exists(seg_dir) else 0)
print("Model dir files:", len(os.listdir(model_dir)) if os.path.exists(model_dir) else 0)
print("Transcript json files:", len(os.listdir(down_trans_dir)) if os.path.exists(down_trans_dir) else 0)

Segmented audio files: 5063
Model dir files: 2
Transcript json files: 105


In [52]:
final_model_dir = os.path.join(BASE_DIR, "04_models", "whisper_small_finetuned_final")
os.makedirs(final_model_dir, exist_ok=True)

trainer.save_model(final_model_dir)
processor.save_pretrained(final_model_dir)

print("Saved final fine-tuned model to:", final_model_dir)
print("Files:", os.listdir(final_model_dir))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved final fine-tuned model to: /content/drive/MyDrive/JoshTalks_ASR/04_models/whisper_small_finetuned_final
Files: ['config.json', 'generation_config.json', 'model.safetensors', 'training_args.bin', 'tokenizer_config.json', 'tokenizer.json', 'processor_config.json']
